# Strategic Thinking  - Predicting Ticket Resolution Delays Across IT Support Queues Using Historical Service Desk Data
**Modules** Data Preparation

This notebook analyses the synthetic dataset (`dataset_jira.xlsx`).
Goals:
1. Clean and prepare the data.
2. Use statistics to describe the data and test relationships.


In [16]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, KFold, StratifiedKFold
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.neighbors import KNeighborsRegressor, KNeighborsClassifier
from sklearn.cluster import KMeans
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, accuracy_score, f1_score, confusion_matrix, silhouette_score

import warnings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

# Part 1 - Data Preparation
## 1.1 Characterisation of the dataset

In [17]:
df = pd.read_excel('dataset_jira.xlsx')

In [18]:
print('Rows (observations):', df.shape[0])
print('Columns (attributes):', df.shape[1])

Rows (observations): 102691
Columns (attributes): 17


In [19]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 102691 entries, 0 to 102690
Data columns (total 17 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   Area                 102691 non-null  object        
 1   Ticket Number        102691 non-null  object        
 2   Created              102691 non-null  datetime64[ns]
 3   Resolved             102691 non-null  datetime64[ns]
 4   Reporter             102686 non-null  object        
 5   Status               102691 non-null  object        
 6   System               102691 non-null  object        
 7   Priority             102689 non-null  object        
 8   SLA Status           102689 non-null  object        
 9   Year                 102689 non-null  float64       
 10  MonthNumber          102689 non-null  float64       
 11  Month                102689 non-null  object        
 12  YearMonth            102689 non-null  datetime64[ns]
 13  MonthYearLabel

In [20]:
print('Missing values per column:')
print(df.isnull().sum())
print()
print('Total missing values:', df.isnull().sum().sum())
print('Duplicate rows:', df.duplicated().sum())

Missing values per column:
Area                   0
Ticket Number          0
Created                0
Resolved               0
Reporter               5
Status                 0
System                 0
Priority               2
SLA Status             2
Year                   2
MonthNumber            2
Month                  2
YearMonth              2
MonthYearLabel         2
Quarter                2
Week                   2
ResolutionTimeHours    2
dtype: int64

Total missing values: 25
Duplicate rows: 0


In [24]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
Year,102689.0,2.025441e+03,4.965348e-01,2025.0,2.025000e+03,2.025000e+03,2.026000e+03,2.026000e+03
MonthNumber,102689.0,6.114384e+00,3.213931e+00,1.0,3.000000e+00,6.000000e+00,9.000000e+00,1.200000e+01
Week,102689.0,2.493579e+01,1.391728e+01,1.0,1.300000e+01,2.500000e+01,3.700000e+01,5.200000e+01
ResolutionTimeHours,102689.0,1.262634e+16,1.229719e+16,38.0,4.105787e+15,8.656378e+15,1.804798e+16,7.994872e+16


In [25]:
 for col in ['Area', 'Status', 'System', 'Priority', 'SLA Status']:
    print(f'\{col}')
    print(df[col].value_counts(dropna=False))
    print('-' * 40)

\Area
JIR - Internal Audit           36792
JIR - Human Resources (HR)     19132
JIR - Accounting               15961
JIR - Financial                 7930
JIR - Legal                     5247
JIR - Administration            4643
JIR - Payrol                    3932
JIR - Compliance                3169
JIR - Service Desk              2914
JIR - IT Support Cloud          2073
JIR - Support                    871
JIR - Commercial Operations       27
Name: Area, dtype: int64
----------------------------------------
\Status
Closed      100974
Resolved      1717
Name: Status, dtype: int64
----------------------------------------
\System
Service General Request with approvals    62346
Incident Management                       22299
Service General Request                   17611
General Request                             435
Name: System, dtype: int64
----------------------------------------
\Priority
Medium    101883
High         500
Low          306
NaN            2
Name: Priority, dtype: i

**Summary**: The dataset contains 102,689 records and 17 variables, including categorical variables (Area, Status, System, Priority and SLA Status), datetime variables (Created and Resolved), and engineered temporal features (Year, MonthNumber, Month, YearMonth, MonthYearLabel, Quarter and Week). The target variable selected for this project is SLA Status, which will be used to identify tickets at risk of breaching Service Level Agreements (SLAs).

A data quality assessment identified 25 missing values distributed across several variables and no duplicate records. Initial analysis revealed a highly imbalanced target variable, with the majority of tickets classified as SLA OK (96,116 records) and a smaller proportion classified as SLA OUT (6,573 records).

The dataset also shows a strong concentration of tickets within the Medium priority category and significant variation in ticket volumes across business areas, with Internal Audit, Human Resources and Accounting generating the highest number of requests. These characteristics will be considered during the data preparation and modelling phases.

## 1.2 Data cleaning and EDA

Remove duplicate records - duplicate tickets would bias the statistics and future predictive models.
Handle missing values - missing data can affect the quality and reliability of the analysis.

Validate date fields - ensure consistency in the Created and Resolved variables before analysis.

Create derived features - generate Year, Month, Quarter and Week to support trend analysis.

Validate the target variable - SLA Status was selected because the project aims to predict SLA breaches.

Check class distribution - identify any imbalance between SLA OK and SLA OUT tickets before modelling.

Validate ResolutionTimeHours - ensure ticket resolution times are calculated correctly for analysis and future predictive modelling.

In [26]:
df['SLA Status'].value_counts(normalize=True) * 100

SLA OK     93.59912
SLA OUT     6.40088
Name: SLA Status, dtype: float64

In [27]:
print(
    round(df['SLA Status'].value_counts(normalize=True)*100,2)
)

SLA OK     93.6
SLA OUT     6.4
Name: SLA Status, dtype: float64


In [28]:
for col in ['Area', 'System', 'Priority', 'Status']:
    print(f"{col}: {df[col].nunique()} unique values")

Area: 12 unique values
System: 4 unique values
Priority: 3 unique values
Status: 2 unique values
